# 蒸馏：让一个小 13 倍的模型模仿 htdemucs

## 蒸馏是什么

**让小模型去学大模型的输出，而不是去学真值。**

| | |
|---|---|
| 教师 | htdemucs，42 M 参数，cSDR 8.80 dB |
| 学生 | 频谱 U-Net，约 3 M 参数（小 13 倍） |
| 训练目标 | 分离得**像教师**，不是像真值 |

## 为什么蒸馏在这个项目里特别划算

正常训分离模型需要**分轨真值** —— 每首歌的人声/鼓/贝斯必须单独存在，
这种数据极稀缺（MUSDB18-HQ 训练集只有 **100 首**）。

蒸馏不需要：**教师跑一遍就把目标造出来了**，所以任何音乐都能当训练数据。
你云盘上有 **5,534 首** Jamendo 音频 —— 比 MUSDB18-HQ 训练集多 **55 倍**。

## 跑之前先写死预测

**学生几乎肯定比教师差。** 蒸馏的产出不是「更好的模型」，
而是帕累托曲线上一个新的点：**更快、稍差**。

| 假设 | 预测 |
|---|---|
| H1 学生能学到教师的大部分能力 | 测试集 cSDR 达到 **7.0 dB 以上**（= 教师的 80%） |
| H2 速度确有优势 | RTF **低于 0.033**（= 教师的一半） |
| H0' 蒸馏在这个规模上不work | cSDR **低于 5 dB**，说明 3 M 参数不够 |

> 若结果落在 5~7 dB，结论是「小模型有能力但需要更多训练/更大 base」，
> 也是有用的信息 —— 但不能说成成功。

## 现实的时间预期

- 可行性验证（第 6 格）：**约 40 分钟**，只用 200 首
- 完整训练（第 7 格）：**8~15 小时**，可断点续训

**先跑验证再决定要不要投一整夜。** 如果 40 分钟后 SDR 曲线没有在往上走，
停掉只亏 40 分钟。


## 0 · 挂载云盘 + 拉代码


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess, sys, time, json

DRIVE = pathlib.Path('/content/drive/MyDrive/Audio AI/MusicMixer')
CKPT_DIR = DRIVE / 'distill_ckpt'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_TAR = DRIVE / 'audio_tar'      # 上一个 notebook 存的 Jamendo 分块

REPO = 'https://github.com/EthanBAI-dev/musicmix.git'
SRC  = pathlib.Path('/content/musicmix')
if SRC.exists():
    subprocess.run(['git', '-C', str(SRC), 'pull', '--ff-only'])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(SRC)])
os.chdir(SRC); sys.path.insert(0, str(SRC))
print('checkpoint 目录:', CKPT_DIR)
print('本地盘可用: %.0f GB' % (shutil.disk_usage('/content').free / 2**30))


## 1 · 装依赖


In [ ]:
!pip -q install demucs soundfile librosa museval

import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if DEV=='cuda' else 'CPU')
assert DEV == 'cuda', '需要 GPU：代码执行程序 → 更改运行时类型 → GPU'


## 2 · 准备训练音频

用云盘上已有的 Jamendo 分块。**只解一两块到本地盘就够训练用了** ——
蒸馏的数据瓶颈不在总量，在教师推理的速度。


In [ ]:
N_CHUNKS = 2          # 解几块到本地（每块约 540 首 / 5 GB）
AUDIO = pathlib.Path('/content/train_audio'); AUDIO.mkdir(exist_ok=True)

tars = sorted(AUDIO_TAR.glob('*.tar'))[:N_CHUNKS] if AUDIO_TAR.exists() else []
if not tars:
    print('⚠️ 云盘上没有 audio_tar/。先跑 extract_features.ipynb（KEEP_AUDIO=True），')
    print('   或手动放一些 mp3 到', AUDIO)
else:
    for t in tars:
        if not any(AUDIO.glob(t.stem[-2:] + '/*.mp3')):
            print('解包', t.name); subprocess.run(['tar', '-xf', str(t), '-C', str(AUDIO)])

FILES = sorted(AUDIO.rglob('*.mp3'))
print(f'训练音频 {len(FILES)} 首')
assert FILES, '没有训练音频'


## 3 · 测试集（用来评学生到底多准）

训练用无标注音乐，**评测必须用有真值的 MUSDB18-HQ**。
只下测试集（50 首），训练集用不到。


In [ ]:
MUSDB = pathlib.Path('/content/musdb18hq')
if not (MUSDB / 'test').exists():
    MUSDB.mkdir(exist_ok=True)
    # MUSDB18-HQ 需要 zenodo 账号；若已在云盘就直接用
    cand = DRIVE / 'musdb18hq_test.tar'
    if cand.exists():
        subprocess.run(['tar', '-xf', str(cand), '-C', str(MUSDB)])
    else:
        print('⚠️ 没找到 MUSDB18-HQ 测试集。')
        print('   把本机的 data/musdb18hq/test 打包成 musdb18hq_test.tar 传到云盘根目录，')
        print('   或跳过评测（第 5 格会自动降级为只看训练损失）。')
TEST_TRACKS = sorted((MUSDB / 'test').glob('*')) if (MUSDB / 'test').exists() else []
print(f'测试曲目 {len(TEST_TRACKS)} 首')


## 4 · 教师、学生、数据

**教师在线生成目标，不预存。** 预存 5,534 首 × 4 轨的分离结果要几百 GB，
而教师推理只占训练时间的一小部分。


In [ ]:
import numpy as np, random, librosa, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from demucs.pretrained import get_model
from demucs.apply import apply_model
from src.separation.student import StudentUNet, distill_loss, SOURCES

SR, SEG = 44100, 5          # 训练片段 5 秒

teacher = get_model('htdemucs').to(DEV).eval()
for p in teacher.parameters(): p.requires_grad_(False)
assert tuple(teacher.sources) == SOURCES, f'声部顺序不一致: {teacher.sources}'

student = StudentUNet(base=32, depth=4).to(DEV)
print(f'教师 {sum(p.numel() for p in teacher.parameters())/1e6:.1f} M  →  '
      f'学生 {student.n_params/1e6:.2f} M   小 '
      f'{sum(p.numel() for p in teacher.parameters())/student.n_params:.1f} 倍')

class RandomSegments(Dataset):
    """每次随机取一首歌的随机 5 秒。不预切片 —— 预切会让同一段被反复看到。"""
    def __init__(self, files, n_per_epoch=512):
        self.files, self.n = files, n_per_epoch
    def __len__(self): return self.n
    def __getitem__(self, i):
        for _ in range(8):                       # 少数文件会解码失败，重试而不是崩
            f = random.choice(self.files)
            try:
                dur = librosa.get_duration(path=str(f))
                if dur < SEG + 1: continue
                off = random.uniform(0, dur - SEG)
                y, _ = librosa.load(str(f), sr=SR, mono=False, offset=off, duration=SEG)
                if y.ndim == 1: y = np.stack([y, y])
                if y.shape[1] < SEG * SR: continue
                return torch.from_numpy(y[:, :SEG*SR].astype('float32'))
            except Exception:
                continue
        return torch.zeros(2, SEG * SR)

loader = DataLoader(RandomSegments(FILES), batch_size=4, num_workers=2, drop_last=True)

@torch.no_grad()
def teacher_targets(mix):
    """教师的分离结果，作为学生的学习目标。"""
    ref = mix.mean(dim=1, keepdim=True)
    m, s = ref.mean(), ref.std() + 1e-8
    out = apply_model(teacher, (mix - m) / s, device=DEV, split=True, overlap=0.1,
                      progress=False)
    return out * s + m


## 5 · 评测（在 MUSDB18-HQ 测试集上）

用 uSDR（全局 SDR）而不是 museval 的 cSDR：museval 很慢（50 首要几十分钟），
训练中途评测需要快。**最终数字仍以本机跑的 museval 为准。**


In [ ]:
import soundfile as sf

def usdr(est, ref, eps=1e-7):
    num = (ref ** 2).sum()
    den = ((ref - est) ** 2).sum()
    return float(10 * np.log10((num + eps) / (den + eps)))

@torch.no_grad()
def evaluate(model, n_tracks=8, seconds=30):
    """返回四轨平均 uSDR。n_tracks 少一点，训练中途要的是**趋势**不是终值。"""
    if not TEST_TRACKS: return None
    model.eval(); scores = []
    for d in TEST_TRACKS[:n_tracks]:
        try:
            mix, _ = librosa.load(str(d / 'mixture.wav'), sr=SR, mono=False,
                                  duration=seconds)
            x = torch.from_numpy(mix.astype('float32'))[None].to(DEV)
            est = model(x)[0].cpu().numpy()
            for i, name in enumerate(SOURCES):
                ref, _ = librosa.load(str(d / f'{name}.wav'), sr=SR, mono=False,
                                      duration=seconds)
                n = min(ref.shape[1], est.shape[2])
                scores.append(usdr(est[i, :, :n], ref[:, :n]))
        except Exception as e:
            print('  评测跳过', d.name, type(e).__name__)
    model.train()
    return float(np.mean(scores)) if scores else None

print('评测函数就绪；测试曲目', len(TEST_TRACKS))


## 6 · 可行性验证（约 40 分钟）

**先花 40 分钟确认曲线在往上走，再决定要不要投一整夜。**

判据很简单：**300 步之后 uSDR 必须明显高于 0 dB 且在上升**。
如果还在 0 附近徘徊，说明架构或学习率有问题，停掉重调，别硬跑。


In [ ]:
import itertools

opt = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=1e-2)
scaler = torch.amp.GradScaler('cuda')

PROBE_STEPS = 300
hist = []
t0 = time.time()
step = 0
for mix in itertools.cycle(loader):
    if step >= PROBE_STEPS: break
    mix = mix.to(DEV)
    with torch.no_grad():
        tgt = teacher_targets(mix)
    with torch.amp.autocast('cuda'):
        loss = distill_loss(student(mix), tgt)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(student.parameters(), 5.0)
    scaler.step(opt); scaler.update()
    step += 1
    if step % 50 == 0:
        sdr = evaluate(student, n_tracks=3, seconds=15)
        hist.append((step, float(loss), sdr))
        print(f'  step {step:>4}  loss {loss.item():.4f}  uSDR {sdr if sdr is None else round(sdr,2)}'
              f'   {(time.time()-t0)/60:.1f} min', flush=True)

print('\n判据：uSDR 应当明显高于 0 且在上升')
if hist and hist[-1][2] is not None:
    first, last = hist[0][2], hist[-1][2]
    print(f'  首次 {first:.2f} dB → 最后 {last:.2f} dB   '
          f'{"✅ 在学" if last > first + 0.5 and last > 0.5 else "⚠️ 没在学，别硬跑"}')


## 7 · 完整训练（8~15 小时，可断点续训）

**随时可以中断。** 每次评测都会存 checkpoint 到云盘，
重跑这一格会自动从最新的 checkpoint 接着走 —— 扛得住 Colab 12 小时会话上限。


In [ ]:
TOTAL_STEPS = 20000
EVAL_EVERY  = 500
CKPT = CKPT_DIR / 'student_base32.pt'

# 断点续训
start_step, best = 0, -99.0
if CKPT.exists():
    ck = torch.load(CKPT, map_location=DEV)
    student.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    start_step, best = ck['step'], ck.get('best', -99.0)
    print(f'从 checkpoint 续训：step {start_step}, 最好 uSDR {best:.2f} dB')

log = []
t0 = time.time(); step = start_step
for mix in itertools.cycle(loader):
    if step >= TOTAL_STEPS: break
    mix = mix.to(DEV)
    with torch.no_grad():
        tgt = teacher_targets(mix)
    with torch.amp.autocast('cuda'):
        loss = distill_loss(student(mix), tgt)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(student.parameters(), 5.0)
    scaler.step(opt); scaler.update()
    step += 1

    if step % EVAL_EVERY == 0:
        sdr = evaluate(student, n_tracks=8, seconds=30)
        el = (time.time() - t0) / 3600
        eta = el / max(step - start_step, 1) * (TOTAL_STEPS - step)
        log.append({'step': step, 'loss': float(loss), 'usdr': sdr})
        mark = ''
        if sdr is not None and sdr > best:
            best = sdr; mark = '  ← 最好'
        # **每次都存**，不只在变好时存 —— 会话被切断时最新状态不能丢
        torch.save({'model': student.state_dict(), 'opt': opt.state_dict(),
                    'step': step, 'best': best, 'log': log,
                    'config': {'base': 32, 'depth': 4}}, CKPT)
        print(f'  step {step:>6}  loss {loss.item():.4f}  '
              f'uSDR {sdr if sdr is None else round(sdr,2)}  '
              f'{el:.1f}h / ETA {eta:.1f}h{mark}', flush=True)

print(f'\n完成。最好 uSDR {best:.2f} dB  →  {CKPT}')


## 8 · 导出给本机做正式评测

Colab 上用的是快速 uSDR。**正式数字要在本机用 museval 跑**（与项目其他数字同口径），
所以这里只导出权重。


In [ ]:
import shutil as sh

if CKPT.exists():
    ck = torch.load(CKPT, map_location='cpu')
    out = CKPT_DIR / 'student_final.pt'
    torch.save({'state_dict': ck['model'], 'config': ck['config'],
                'step': ck['step'], 'best_usdr': ck['best'], 'log': ck['log']}, out)
    mb = out.stat().st_size / 2**20
    print(f'✅ → {out}  ({mb:.1f} MB)  step {ck["step"]}  最好 uSDR {ck["best"]:.2f} dB')
    print('\n本机下载后跑：')
    print('  python -m scripts.run_separation_eval --model student '
          '--ckpt results/student_final.pt --subset test --out results/p8_student.md')
else:
    print('还没有 checkpoint')


## 9 · 结果怎么解读

对照第 0 格写下的预测：

| 实测 uSDR | 结论 |
|---|---|
| ≥ 7.0 dB | **H1 成立** —— 学生学到了教师的大部分能力，帕累托曲线上多一个有效点 |
| 5.0 ~ 7.0 | 部分成立 —— 小模型有能力但欠训练或 base 太小。**不能说成成功** |
| < 5.0 dB | **H0' 成立** —— 3 M 参数在这个任务上不够 |

无论哪个结果都要写进 `results/`，并在帕累托图上加这个点。
**一个更快但更差的模型是有价值的数据点，一个被掩盖的失败不是。**

> 注意 Colab 的 uSDR 与本机的 cSDR **口径不同**，数值不可直接比较。
> 拿去和 8.80 dB 对比之前，必须在本机用 museval 重跑。
